# Altimetry time series for Terschelling

## Paths and settings

In [1]:
path_output_general = '/media/bene/Seagate/PhD-data/98_paper2_kf_dtm/4tu/output/' # general output for all study areas
path_output = '/media/bene/Seagate/PhD-data/98_paper2_kf_dtm/4tu/output/Ters/' # output only for Terschelling
path_input_general = '/media/bene/Seagate/PhD-data/98_paper2_kf_dtm/4tu/input/' # general input for all study areas
path_input = '/media/bene/Seagate/PhD-data/98_paper2_kf_dtm/4tu/input/Ters/' # input only for Terschelling

epsg = {'in':4326, 'out':28992} # 32631 28992
overwrite = False # True: overwrite existing files, False: do not overwrite

## Imports

In [2]:
%load_ext autoreload
%autoreload 2

import os
import xarray as xr
import numpy as np
import pandas as pd
    
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
from cartopy.feature import LAND

from coastal_data import CD_helper_functions, CD_altimetry, CD_data_preparation

import pdb

## Data

### IB correction

In [3]:
# Download mean sea level pressure from ERA5
fn = 'era5_mean_sea_level_pressure.nc'
if (not os.path.isfile(path_output_general+fn)) | overwrite:
    CD_data_preparation.download_era5_mean_sea_level_pressure(path_output_general, fn)

In [4]:
mslp = xr.open_dataset(path_output_general + 'era5_mean_sea_level_pressure.nc')

In [5]:
def apply_ocean_mask_to_era5msl(path_input_general, mslp):
    mask = xr.open_dataset(path_input_general + 'land_mask_0p25_oceanFull_invGrd.nc')
    mask = mask.squeeze('time').reset_coords('time', drop=True)
    mask = mask.z > 0.5
    mask = mask[:,:1440]
    mask = np.flipud(mask)
    mask_temp1 = mask[:,:720]
    mask_temp2 = mask[:,720:]
    mask = np.hstack((mask_temp2, mask_temp1))
    mask = np.invert(mask)
    mask = np.broadcast_to(mask, mslp.msl.shape)
    msl_ocean = np.ma.masked_array(mslp.msl, mask=mask)

    return msl_ocean, mask

In [6]:
msl_ocean, mask = apply_ocean_mask_to_era5msl(path_input_general, mslp)

In [7]:
def latitude_weighting(ds, mask):
    weights = np.cos(ds.latitude * np.pi / 180)
    weights = np.resize(weights, (len(ds.longitude),len(ds.latitude)))
    weights = np.transpose(weights)
    weights = np.ma.masked_array(weights, mask=mask[0])

    return weights

In [8]:
weights = latitude_weighting(mslp, mask)

In [ ]:
msl_ocean_w = msl_ocean * weights

In [ ]:
msl_ocean_w_mean = msl_ocean_w.sum(axis=(1,2)) / weights.sum()

### PSMSL Tide gauge

In [ ]:
fn = 'psmsl_monthly_RLR_TERS.txt'
tg_psmsl_orig = pd.read_csv(f'{path_input}/2_sea_level/{fn}', sep=';', header=None, names=['date', 'SSH_RLR[mm]'])

tg_psmsl = pd.DataFrame()
tg_psmsl['date'] = CD_helper_functions.decimal_numbers_to_datetime(tg_psmsl_orig['date'])
tg_psmsl = tg_psmsl.set_index('date')

# translate RLR to NAP (numbers from https://psmsl.org/data/obtaining/rlr.diagrams/236.php)
idx = np.nonzero(tg_psmsl.index.year < 2005)[0]
tg_psmsl.loc[tg_psmsl.index[idx], 'SSH_NAP[cm]'] = ((tg_psmsl_orig.loc[tg_psmsl_orig.index[idx], 'SSH_RLR[mm]'] - 6936) / 10).values

idx = np.nonzero(tg_psmsl.index.year >= 2005)[0]
tg_psmsl.loc[tg_psmsl.index[idx],'SSH_NAP[cm]'] = ((tg_psmsl_orig['SSH_RLR[mm]'][idx] - 6957) / 10).values

# reduce to altimetry time period
tg_psmsl = tg_psmsl.iloc[np.nonzero(tg_psmsl.index.year >= 1992)[0]]

# IB correction


In [ ]:
tg_psmsl

### OpenADB ALES

In [ ]:
# Merge all .nc-files
fn = 'ales.nc'
if (not os.path.isfile(path_output + fn)) | overwrite:
    ales = CD_altimetry.combine_openadb_nc_files(f'{path_input}2_sea_level/2_OpenADB_Ters')
    ales.to_netcdf(path_output + fn)
else:
    ales = xr.open_dataset(path_output + fn)

In [ ]:
# Select which missions to use
missions = [
            'Envisat (Version 3)', # 21.06.2002 - 11.10.2010
            'Jason-1',             # 15.01.2002 - 14.01.2009
            # 'Jason-1 (EM)',      # 10.02.2009 - 02.03.2012
            # 'Jason-1 (GM)',      # 11.05.2012 - 17.06.2013
            'Jason-2',             # 12.07.2008 - 30.09.2016
            # 'Jason-2 (EM)',      # 15.10.2016 - 17.05.2017
            'Jason-3',             # 18.02.2016 - 20.04.2019
            'Saral',               # 17.03.2013 - 28.06.2016
            # 'Saral (DP)',        # 05.07.2016 - 14.04.2019
            'Sentinel-3A',         # 24.11.2017 - 04.04.2020
            'Sentinel-3B',         # 26.11.2018 - 12.04.2020
           ]
ales = CD_altimetry.select_missions(ales, missions)

# Clean according to OpenADB instructions (e.g. dist to coast, std, ...)
ales = CD_altimetry.clean_openadb_ales(ales)

Cells extracted for ALES: 88 87 76 75 64 63 (epsg 28992)  
96 95 83 82 70 69 (epsg 32631)

In [ ]:
# Make/load timeseries
fn = f'ales_ts_epgs{epsg['out']}.csv'
if (not os.path.isfile(path_output+fn)) | overwrite:
    labels = {'time':'time', 'lon':'glon', 'lat':'glat', 'sla':None, 'ssh':'ssh', 'mssh':'mssh'}
    tg = tg_psmsl['ssh_ib_vlm']
    gridsize = {'x':25, 'y':25} # [km]
    period_covered = {'min':'2002-07-01', 'max':'2020-03-31'}
    freq_average = 'ME' # period over which to average in pd.Grouper
    # 'ME': month end frequency
    # https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#offset-aliases
    %matplotlib qt5
    ales_ts = CD_altimetry.get_altimetry_timeseries_with_TG(ales, labels, epsg, tg, gridsize, period_covered, freq_average)
    ales_ts.to_csv(path_output+fn)
else:
    ales_ts = pd.read_csv(path_output+fn, index_col='time', parse_dates=['time'])

### RADS T/P

In [ ]:
path_tp = f'{path_input}2_sea_level/1_marge_topexA_ters'
tx_018 = xr.open_dataset(path_tp+'rads2nc_pass018.nc')
tx_137 = xr.open_dataset(path_tp+'rads2nc_pass137.nc')
tx_196 = xr.open_dataset(path_tp+'rads2nc_pass196.nc')

tx_combined = xr.merge([tx_018, tx_137, tx_196])
tx = CD_altimetry.clean_rads(tx_combined)

Cells extracted for T/P: 45 46 55 54 64 63 (epsg 28992 and epsg 32631) would be the best fitting cells  
80 79 69 68 67 57 would be closer to selection from ALES

In [ ]:
# Make/load timeseries
fn = f'tx_ts_epsg{epsg['out']}.csv'
(if not os.path.isfile(path_output+fn)) | overwrite:
    labels = {'time':'time', 'lon':'lon', 'lat':'lat', 'sla':'sla', 'ssh':None, 'mssh':'mss'}
    tg = tg_psmsl['ssh_ib_vlm']
    gridsize = {'x':25, 'y':25} # [km]
    period_covered = {'min':'1993-01-31', 'max':'2002-03-31'}
    freq_average = 'ME' # period over which to average in pd.Grouper
    # 'ME': month end frequency
    # https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#offset-aliases
    %matplotlib qt5
    tx_ts = CD_altimetry.get_altimetry_timeseries_with_TG(tx, labels, epsg, tg, gridsize, period_covered, freq_average)
    tx_ts.to_csv(path_output+fn)
else:
    tx_ts = pd.read_csv(path_output+fn, index_col='time', parse_dates=['time'])

## Remove bias between OpenADB and RADS timeseries

In [ ]:
def equalise_timeseries(ts1, ts2):
    '''
    From two timeseries ts1 and ts2 with some overlap
    compute the bias from the overlap period as the mean of differences,
    and adjust ts1 so that both are on the average level of ts2.

    Input
    -----
    ts1, ts2: pandas Series with DateTime index

    Output
    -----
    ts1 with bias to ts2 removed
    '''
    overlap_dates = ts1.index.intersection(ts2.index)
    print(f'There are {len(overlap_dates)} overlapping values in the same time period.')
    
    idx_ts1 = ts1.index.get_indexer(overlap_dates)
    ts1_overlap = ts1.iloc[idx_ts1]

    idx_ts2 = ts2.index.get_indexer(overlap_dates)
    ts2_overlap = ts2.iloc[idx_ts2]

    bias = (ts1_overlap - ts2_overlap).mean()
    print(f'Bias: {bias} m')
    ts1_biased = ts1 - bias
    # Remove the overlapping dates from ts1
    ts1_biased = ts1_biased.drop(overlap_dates)
    
    return ts1_biased

In [ ]:
tx_biased_to_ales = equalise_timeseries(tx_ts.sla_temp_av, ales_ts.sla_temp_av)

In [ ]:
combined = pd.concat([tx_biased_to_ales, ales_ts.sla_temp_av])

In [ ]:
fn = f'tp_plus_ales_epsg{epsg['out']}.csv'
(if not os.path.isfile(path_output+fn)) | overwrite:
    combined.to_csv(path_output+fn)